# Práctica: Chat de preguntas frecuentes con PLN · PitLineMotor

En este cuaderno construimos un asistente sencillo que responde a las dudas de los clientes de un taller mecánico. El cliente escribe su pregunta con sus propias palabras y el asistente busca, entre las preguntas que conoce, la que más se le parece.

---

## Objetivos
1. **Preprocesamiento:** pasar a minúsculas, quitar signos, tokenizar, eliminar *stop words* y reducir cada palabra a su raíz (*stemming*).
2. **Representación del texto:** convertir las preguntas en vectores con `CountVectorizer` y `TfidfVectorizer`.
3. **Búsqueda por similitud:** comparar la pregunta del cliente con las conocidas mediante la **similitud del coseno**.
4. **Evaluación y prueba:** medir cuántas preguntas nuevas acierta y probarlo como un chat.

---

**Análisis del fallo:** a *«¿Hacéis cambio de aceite?»* responde cómo cambiar el email, porque comparte la raíz *cambi* con *«¿Cómo cambio mi email?»*. El modelo compara palabras, no entiende su significado. Se corrige añadiendo frases a la base de conocimiento o subiendo el umbral, a costa de que deje de entender algunas preguntas válidas.

In [ ]:
# ============================================================
# Chat de preguntas frecuentes con PLN · PitLineMotor
# ============================================================
import nltk
import re
import pandas as pd

# 1. Recursos de NLTK (solo se descargan la primera vez)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 2. Base de conocimiento: varias formas de preguntar lo mismo, por tema
preguntas = {
    'Pedir cita': [
        "¿Cómo pido cita?",
        "Quiero reservar una cita para mi coche",
        "¿Puedo pedir hora para llevar el vehículo al taller?"],
    'Estado de la reparación': [
        "¿Cómo va la reparación de mi coche?",
        "¿En qué fase está mi vehículo?",
        "Quiero saber si ya han arreglado mi coche"],
    'Factura': [
        "¿Dónde descargo la factura?",
        "Necesito la factura de la reparación",
        "¿Cómo consigo el PDF de la factura?"],
    'Pendiente de pieza': [
        "¿Por qué mi reparación está parada?",
        "¿Qué significa pendiente de pieza?",
        "Mi coche está esperando una pieza"],
    'Recoger el coche': [
        "¿Cuándo puedo recoger mi coche?",
        "¿Ya puedo ir a por el vehículo?",
        "Quiero recoger el coche del taller"],
    'Precio': [
        "¿Cuánto cuesta la mano de obra?",
        "¿Qué precio tiene la hora de trabajo?",
        "¿Cómo se calcula el precio de la reparación?"],
    'Cambiar email': [
        "¿Cómo cambio mi email?",
        "Quiero modificar mi correo electrónico",
        "Actualizar el correo de mi cuenta"],
    'Historial': [
        "¿Puedo ver reparaciones anteriores?",
        "Historial de reparaciones de mi coche",
        "¿Qué le hicieron a mi coche la última vez?"],
}

respuestas = {
    'Pedir cita': "Entra en «Mis citas» y crea una cita nueva con tu vehículo, la fecha y el motivo. Quedará pendiente hasta que Recepción la confirme.",
    'Estado de la reparación': "En «Mis reparaciones» puedes ver en qué punto está cada orden: diagnosis, reparación, pendiente de pieza, prueba o finalizada.",
    'Factura': "Cuando la reparación está finalizada, entra en «Mis reparaciones» y descarga la factura en PDF.",
    'Pendiente de pieza': "Falta una pieza para continuar. El taller ya la ha pedido y, cuando llegue, el mecánico reanudará la reparación.",
    'Recoger el coche': "Cuando la reparación aparece como finalizada, el coche está listo. Pasa por Recepción, que registrará la entrega.",
    'Precio': "Se cobra el tiempo de diagnosis y reparación al precio por hora del taller, más las piezas utilizadas y el IVA. Todo aparece desglosado en la factura.",
    'Cambiar email': "Puedes cambiar tu email desde «Mi perfil».",
    'Historial': "En «Mis reparaciones» aparecen también las reparaciones anteriores con todos sus datos.",
}

df = pd.DataFrame([(p, tema) for tema, lista in preguntas.items() for p in lista],
                  columns=['pregunta', 'tema'])

# 3. Preprocesamiento: minúsculas, sin signos, tokens, sin stop words (salvo "no") y raíz
stemmer = SnowballStemmer('spanish')
stop_words_es = set(stopwords.words('spanish')) - {'no'}

def preprocesar(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)
    tokens = word_tokenize(texto, language='spanish')
    return " ".join(stemmer.stem(t) for t in tokens if t not in stop_words_es)

df['pregunta_limpia'] = df['pregunta'].apply(preprocesar)

# 4. Vectorización con TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['pregunta_limpia'])

# 5. Modelo: buscar la pregunta más parecida (similitud del coseno)
UMBRAL = 0.3

def responder(pregunta):
    vector = vectorizer.transform([preprocesar(pregunta)])
    similitudes = cosine_similarity(vector, X)[0]
    mejor = similitudes.argmax()
    if similitudes[mejor] < UMBRAL:
        return None, similitudes[mejor], "No te he entendido. Prueba a decirlo de otra forma o pregunta en Recepción."
    tema = df.loc[mejor, 'tema']
    return tema, similitudes[mejor], respuestas[tema]

# 6. Evaluación con preguntas nuevas
pruebas = [
    ("Me gustaría reservar hora para el lunes", 'Pedir cita'),
    ("¿Ya han terminado de arreglar mi vehículo?", 'Estado de la reparación'),
    ("No encuentro la factura de mi coche", 'Factura'),
    ("Me dicen que falta una pieza, ¿qué pasa?", 'Pendiente de pieza'),
    ("¿A qué hora paso a recoger el coche?", 'Recoger el coche'),
    ("¿Cuánto cuesta la reparación?", 'Precio'),
    ("Quiero cambiar mi correo", 'Cambiar email'),
    ("¿Puedo ver las reparaciones que le hicisteis antes?", 'Historial'),
    ("¿Abrís los domingos?", None),
    ("¿Vendéis neumáticos?", None),
    ("¿Hacéis cambio de aceite?", None),
]

aciertos = 0
print("=== Evaluación ===")
for texto, esperado in pruebas:
    tema, similitud, _ = responder(texto)
    ok = tema == esperado
    aciertos += ok
    print(f"{'✔' if ok else '✘'} {texto:<52} → {tema or 'Sin respuesta'} ({similitud:.2f})")
print(f"Aciertos: {aciertos} de {len(pruebas)} ({aciertos / len(pruebas) * 100:.0f} %)\n")

# 7. Chat con el asistente
print("Asistente PitLineMotor: Hola, ¿en qué te puedo ayudar? (escribe 'salir' para terminar)")
while True:
    pregunta = input("Tú: ")
    if pregunta.strip().lower() == 'salir':
        print("Asistente: ¡Hasta luego!")
        break
    tema, similitud, respuesta = responder(pregunta)
    print("Asistente:", respuesta)
    print(f"   (tema: {tema or 'ninguno'} · similitud: {similitud:.2f})")

=== Evaluación ===
✔ Me gustaría reservar hora para el lunes              → Pedir cita (0.46)
✔ ¿Ya han terminado de arreglar mi vehículo?           → Estado de la reparación (0.40)
✔ No encuentro la factura de mi coche                  → Factura (0.46)
✔ Me dicen que falta una pieza, ¿qué pasa?             → Pendiente de pieza (0.61)
✔ ¿A qué hora paso a recoger el coche?                 → Recoger el coche (0.52)
✔ ¿Cuánto cuesta la reparación?                        → Precio (0.64)
✔ Quiero cambiar mi correo                             → Cambiar email (0.48)
✔ ¿Puedo ver las reparaciones que le hicisteis antes?  → Historial (0.66)
✔ ¿Abrís los domingos?                                 → Sin respuesta (0.00)
✔ ¿Vendéis neumáticos?                                 → Sin respuesta (0.00)
✘ ¿Hacéis cambio de aceite?                            → Cambiar email (0.64)
Aciertos: 10 de 11 (91 %)

Asistente PitLineMotor: Hola, ¿en qué te puedo ayudar? (escribe 'salir' para terminar)
